## Thread Synchronization

Unlike CPU programming (C/C++), CUDA introduces additional complexity because GPU operations execute asynchronously across multiple CUDA streams and thousands of GPU threads.

**1. What is thread synchronization in CUDA?**

**Thread synchronization** ensures that multiple GPU threads or operations execute in the required order before proceeding to the next operation.

CUDA has several levels of synchronization:

| Synchronization | Purpose |
| :--- | :--- |
| `torch.cuda.synchronize()` | Waits for all previously queued GPU work on the current device |
| `torch.cuda.Stream.synchronize()` | Waits for work in a specific stream |
| `torch.cuda.Event` | Synchronizes or tracks execution between GPU operations |
| `__syncthreads()` | Synchronizes threads within a CUDA thread block |
| `torch.cuda.wait_event()` | Makes a stream wait for a CUDA event |
| `torch.cuda.wait_stream()` | Makes one stream wait for another stream |

**Important:** CUDA operations are generally asynchronous when launched from the CPU.
For example:

```python
import torch

x = torch.randn(1000, 1000, device="cuda")
y = torch.randn(1000, 1000, device="cuda")

z = torch.matmul(x, y)

print("Execution completed")
```

The CPU may print `"Execution completed"` before the GPU finishes matrix multiplication.

To wait for GPU completion:

```python
torch.cuda.synchronize()

print("GPU execution completed")
```


## CUDA deadlock

A **deadlock** occurs when two or more threads, streams, or processes wait indefinitely for resources or synchronization conditions that cannot be satisfied.

Consider this conceptual example:

```text
Deadlock scenario

+-----------------------------+   +-----------------------------+

| Thread A                    |   | Thread B                    |
| Holds Resource 1            |   | Holds Resource 2            |
| ↓                           |   | ↓                           |
| Waits for Resource 2        |   | Waits for Resource 1        |
+-----------------------------+   +-----------------------------+

[Deadlock] Neither thread can proceed.
```

**3. Common CUDA deadlock and synchronization scenarios**

| Scenario | Problem | Possible symptom |
| :--- | :--- | :--- |
| Incorrect stream synchronization | Streams wait on each other | GPU execution hangs |
| Incorrect CUDA events | Waiting for an event that is never recorded | Indefinite waiting |
| Incorrect `__syncthreads()` usage | Threads in a block do not reach the barrier consistently | Kernel hangs or undefined behavior |
| CPU-GPU synchronization | CPU waits for GPU while GPU work depends on CPU | Application hangs |
| Multiprocessing synchronization | Processes wait on each other's locks or queues | Training hangs |
| Distributed training | One GPU waits for another GPU's collective operation | DDP training hangs |
| Resource contention | Multiple operations compete for resources | Performance degradation or stalls |

#### CUDA stream:

`torch.cuda.stream()` is a PyTorch context manager used to execute CUDA operations on a specific CUDA stream instead of the default stream.

### Syntax

```python
stream = torch.cuda.Stream()

with torch.cuda.stream(stream):
    # CUDA operations run on this stream
    y = model(x)
```

##### What is a CUDA Stream?

A CUDA stream is a sequence of GPU operations that execute in order.

* Same stream (Single stream) → operations execute sequentially.
* Different streams (Multiple Streams) → operations may execute concurrently (parallel execution) if GPU resources allow.

Default behavior:

```python
y = a + b
z = y * c
```

Both operations are submitted to the **default stream**.

#### Pipeline Inference, Multiple Model run

```python
stream1 = torch.cuda.Stream()
stream2 = torch.cuda.Stream()

with torch.cuda.stream(stream1):
    out1 = model(batch1)

with torch.cuda.stream(stream2):
    out2 = model(batch2)
```

Can improve GPU utilization.

#### Custom CUDA Workloads Useful in:

* Multi-model inference
* LLM serving systems
* High-throughput pipelines
* Distributed training
* CUDA Graphs


#### Example:
Check CUDA DeadLock example. First we Create Two CUDA Streams, Create Input Tensors & tensors are allocated on the GPU. After that we Submit first & second Matrix Multiplication using stream 1 & 2.

- Below example No Deadlock occurs in this example:
    - PyTorch supports multiple CUDA streams to execute GPU operations concurrently.

        ```python
        import torch

        stream1 = torch.cuda.Stream()
        stream2 = torch.cuda.Stream()

        x = torch.randn(1000, 1000, device="cuda")
        y = torch.randn(1000, 1000, device="cuda")

        with torch.cuda.stream(stream1):
            a = torch.matmul(x, y)

        with torch.cuda.stream(stream2):
            b = torch.matmul(x, y)

        torch.cuda.synchronize()
        ```
    Here, both streams can execute independently.

    - However, suppose stream2 needs the output of stream1. You must establish an execution dependency.

        ```python
        stream1 = torch.cuda.Stream()
        stream2 = torch.cuda.Stream()

        with torch.cuda.stream(stream1):
            a = torch.matmul(x, y)
            event = torch.cuda.Event()
            event.record()

        with torch.cuda.stream(stream2):
            stream2.wait_event(event)
            b = torch.matmul(a, y)

        torch.cuda.synchronize()
        ```

    - This ensures that stream2 does not use `a` before stream1 finishes producing it.

### torch.cuda.Stream and torch.cuda.stream:

- torch.cuda.Stream() creates a new CUDA stream.
- torch.cuda.stream(s) tells PyTorch to enqueue operations into that stream.

### Same Stream vs Different Streams

- Same Stream

    ```python
    with torch.cuda.stream(s1):
        op1()
        op2()
    ```    
    - Timeline:
        ```text
        0ms     5ms     10ms

        op1 =====
                op2 =====
        ```
    - Total ≈ 10 ms

- Different Streams

    ```python
    with torch.cuda.stream(s1):
        op1()

    with torch.cuda.stream(s2):
        op2()
    ```    
    - Timeline:
    ```text
    0ms     5ms
    op1 =====
    op2 =====
    ```
    - Total ≈ 5 ms (if fully overlapped)


**How to debug CUDA hangs and synchronization issues?**

Useful PyTorch debugging techniques:

| Technique | Purpose |
| :--- | :--- |
| `torch.cuda.synchronize()` | Identify asynchronous execution errors |
| `CUDA_LAUNCH_BLOCKING=1` | Makes CUDA launches synchronous for debugging |
| `torch.cuda.memory_summary()` | Inspect GPU memory allocation |
| `torch.cuda.memory_allocated()` | Check currently allocated GPU memory |
| `torch.cuda.memory_reserved()` | Check memory reserved by PyTorch |
| `torch.autograd.set_detect_anomaly(True)` | Detect certain autograd errors |
| `compute-sanitizer` | Detect CUDA memory and synchronization errors in supported CUDA applications |

For example, enable synchronous CUDA execution:

```bash
CUDA_LAUNCH_BLOCKING=1 python train.py
```
- What CUDA_LAUNCH_BLOCKING=1 Does?
    - When enabled every CUDA operation becomes synchronous. This helps identify the actual operation causing an asynchronous CUDA error. It is a debugging technique, not a production optimization. Default CUDA operations are queued asynchronously after CUDA_LAUNCH_BLOCKING=1 every CUDA operation becomes synchronous.


## torch.cuda.synchronize()

- Q. Can i assume torch.cuda.synchronize() only do wait for GPU completed tensor task?

    - Yes, that is a correct way to think about it. `torch.cuda.synchronize()` blocks the CPU thread and waits until **all previously queued CUDA operations (tensor computations, kernel launches, memory copies, etc.) on the GPU are fully completed**. After `torch.cuda.synchronize()` returns, you can safely assume that every GPU task submitted before the call has finished execution.
    - However, `torch.cuda.synchronize()` does **not** synchronize processes, threads, or Python code. It only synchronizes the CPU with GPU work. In simple terms, **"wait until the GPU has completed all pending tensor-related operations, then continue CPU execution."**


## why Asynchronous CUDA?

CUDA is **asynchronous by design** because it allows the CPU and GPU to work simultaneously instead of making one wait for the other.

- **Let imagine If CUDA Were Synchronous:**
    ```python
    a = torch.matmul(x, y)
    b = torch.matmul(a, z)
    c = torch.matmul(b, w)
    ```
- Synchronous execution:
    ```text
    CPU: Launch Kernel 1
    CPU: Wait

    GPU: Execute Kernel 1

    CPU: Launch Kernel 2
    CPU: Wait

    GPU: Execute Kernel 2

    CPU: Launch Kernel 3
    CPU: Wait

    GPU: Execute Kernel 3
    ```
- Timeline:
    ```text
    CPU  █░░░█░░░█░░░
    GPU  ░███░███░███
    ```
    - Lots of idle time.

- **Actual CUDA Behavior (Asynchronous)**
    - When PyTorch launches a kernel:
        ```python
        a = torch.matmul(x, y)
        ```
    - the CPU only does:
        ```text
        "GPU, please execute this later."
        ```
    - and immediately continues.
        ```text
        CPU
        ├── Launch Kernel1
        ├── Launch Kernel2
        ├── Launch Kernel3
        └── Continue
        ```
    - while:
        ```text
        GPU
        ├── Execute Kernel1
        ├── Execute Kernel2
        └── Execute Kernel3
        ```
    - Timeline:
        ```text
        CPU  ███
        GPU    █████████
        ```
    - Much more efficient.



## Synchronization using barriers & DDP synchronization

#### Synchronization using barriers:

**Scenario:** Three processes execute different GPU workloads, but all must finish their preparation phase before moving to the next phase.

**Barrier synchronization** is a synchronization technique in which multiple threads or processes must reach a specific execution point (barrier) before any of them can continue. A thread or process that reaches the barrier earlier waits until all participating threads or processes arrive. Once everyone reaches the barrier, execution continues.

In PyTorch, barriers are useful in multiprocessing and distributed training when multiple workers must complete one stage before starting the next. For example, during distributed model training, all GPU workers may need to finish gradient computation before proceeding to the next training stage. PyTorch provides `torch.distributed.barrier()` for this purpose. For CPU threads, Python provides `threading.Barrier()`.


```python
    print(f"Process {rank}: Starting GPU operation")

    y = x @ x

    torch.cuda.synchronize()

    print(f"Process {rank}: GPU completed")

if __name__ == "__main__":

    mp.set_start_method("spawn", force=True)

    barrier = mp.Barrier(3)

    mp.spawn(
        worker,
        args=(barrier,),
        nprocs=3,
        join=True
    )
```
A barrier creates a synchronization point between CPU processes.

**Important distinction:** The barrier ensures that all processes have reached the barrier. It does not guarantee that GPU operations launched before the barrier have completed. That is why the example synchronizes GPU execution before reporting completion.


In [2]:
import threading
import time

barrier = threading.Barrier(3)

def worker(worker_id):

    print(f"Worker {worker_id}: Starting")

    time.sleep(worker_id)

    print(f"Worker {worker_id}: Reached barrier")

    barrier.wait()

    print(f"Worker {worker_id}: Continuing execution")


threads = [
    threading.Thread(target=worker, args=(i,))
    for i in range(1, 4)
]

for t in threads:
    t.start()

for t in threads:
    t.join()

Worker 1: Starting
Worker 2: Starting
Worker 3: Starting
Worker 1: Reached barrier
Worker 2: Reached barrier
Worker 3: Reached barrier
Worker 3: Continuing execution
Worker 1: Continuing execution
Worker 2: Continuing execution


**How it works:**

* Three threads start execution independently.
* Each thread performs its task.
* When a thread reaches `barrier.wait()`, it waits.
* Once all three threads reach the barrier, all can continue execution.

**2. Difference between synchronization techniques**

| Technique | Purpose | Behavior |
| :--- | :--- | :--- |
| `threading.Lock()` | Protect shared resources | Only one thread enters critical section |
| `torch.multiprocessing.Lock()` | Protect shared process resources | Only one process enters critical section |
| `threading.Event()` | Signal an event | Waiting threads continue when event is set |
| `threading.Semaphore()` | Control resource access | Allows a limited number of threads |
| `threading.Barrier()` | Synchronize execution stages | All participants wait until everyone arrives |
| `torch.cuda.synchronize()` | Synchronize GPU execution | CPU waits for previously submitted GPU work |

#### Barrier synchronization in PyTorch distributed training

#### Q. What is Distributed training?

**Distributed training** is a deep learning technique in which training computation is distributed across multiple GPUs or machines instead of using a single GPU. Each GPU processes a portion of the training data, performs forward and backward propagation, and participates in gradient synchronization. This reduces training time and allows us to train larger models and process larger datasets. For example, suppose you have 4 GPUs and a batch size of 128. Each GPU processes 32 samples, calculates gradients independently, and uses DDP (Distributed Data Parallel) All-Reduce to synchronize gradients before updating model parameters.

**Why is Barrier synchronization required?**

In distributed training, multiple processes execute independently, and some processes may finish their tasks earlier than others. **Barrier synchronization ensures that all participating processes reach a specific execution point before any of them continues.** For example, suppose four GPU processes finish training and need to start model validation. GPU 0 may finish earlier, while GPU 3 is still processing its batch. A barrier makes GPU 0's process wait until all four processes reach the barrier. This prevents one process from proceeding to the next stage while others are still completing the previous stage.


PyTorch provides:

```python
torch.distributed.barrier()
```

Example:

```python
import torch.distributed as dist

# Each distributed worker completes its task
train_one_epoch()

# Wait until all workers reach this point
dist.barrier()

# Continue after synchronization
validate_model()
```
Here, `dist.barrier()` ensures that all participating distributed processes reach the synchronization point before proceeding.

**4. When should you use barriers?**

* **Multi-threading:** Wait for all threads to finish data preparation before starting computation.
* **Multiprocessing:** Ensure all processes complete dataset preparation before training.
* **Distributed training:** Synchronize multiple GPU workers between training stages.
* **CUDA:** Use CUDA events and stream synchronization for GPU execution dependencies; a CPU thread barrier alone does not synchronize GPU kernels.

**Interview takeaway:** A Lock protects shared resources, an Event communicates that something has happened, a Semaphore controls the number of concurrent users, and a Barrier synchronizes multiple threads or processes at a common execution point. The key difference is that **a Barrier waits for all participating workers, whereas a Lock provides exclusive access to a critical section.**


# DDP Synchronization in PyTorch

## 1. What is DDP?

**Distributed Data Parallel (DDP)** is a PyTorch technique for training
one model across multiple GPUs using multiple processes. Each process
owns a replica of the model and processes a different portion of the
training data.

During backpropagation, DDP automatically synchronizes gradients across
processes using **All-Reduce** communication. Each process receives the
averaged gradients and applies the same optimizer update. When model
initialization, data handling, optimizer state, and update steps are
consistent, the model replicas remain synchronized.

DDP is commonly used for multi-GPU training because it performs
computation independently on each GPU while coordinating gradient
updates.

## 2. Distributed Data Parallel architecture

The following Mermaid diagram shows a typical four-GPU DDP training
iteration.

``` mermaid
flowchart TB
    D["Training dataset"] --> S["DistributedSampler"]
    S --> B0["Batch for rank 0"]
    S --> B1["Batch for rank 1"]
    S --> B2["Batch for rank 2"]
    S --> B3["Batch for rank 3"]

    subgraph P0["Process 0"]
        B0 --> G0["GPU 0 model replica"]
        G0 --> F0["Forward pass"]
        F0 --> L0["Loss"]
        L0 --> BW0["Backward pass"]
        BW0 --> GR0["Local gradients"]
    end

    subgraph P1["Process 1"]
        B1 --> G1["GPU 1 model replica"]
        G1 --> F1["Forward pass"]
        F1 --> L1["Loss"]
        L1 --> BW1["Backward pass"]
        BW1 --> GR1["Local gradients"]
    end

    subgraph P2["Process 2"]
        B2 --> G2["GPU 2 model replica"]
        G2 --> F2["Forward pass"]
        F2 --> L2["Loss"]
        L2 --> BW2["Backward pass"]
        BW2 --> GR2["Local gradients"]
    end

    subgraph P3["Process 3"]
        B3 --> G3["GPU 3 model replica"]
        G3 --> F3["Forward pass"]
        F3 --> L3["Loss"]
        L3 --> BW3["Backward pass"]
        BW3 --> GR3["Local gradients"]
    end

    GR0 --> AR["All-Reduce gradient synchronization"]
    GR1 --> AR
    GR2 --> AR
    GR3 --> AR

    AR --> AVG["Averaged gradients on each rank"]
    AVG --> O["Each process runs optimizer.step()"]
    O --> N["Synchronized model replicas for next iteration"]
```

### Architecture explanation

1.  **Model replication:** Each process creates the same model
    architecture and DDP initializes model parameters consistently
    across ranks.
2.  **Data distribution:** `DistributedSampler` assigns each process a
    different subset of the dataset.
3.  **Forward pass:** Each GPU runs its local model replica on its local
    mini-batch.
4.  **Backward pass:** Each process computes gradients from its local
    loss.
5.  **Gradient synchronization:** DDP uses All-Reduce to aggregate
    gradients across processes and make the averaged gradients available
    to each process.
6.  **Optimizer update:** Each process applies the same optimizer update
    to its model replica.

For the usual DDP setup, gradients are synchronized during
`loss.backward()`. DDP uses gradient buckets to overlap communication
with backward computation where possible.

## 3. Example: DDP training with two or more GPUs

Save this as `ddp_train.py`. Run it using `torchrun`, not by starting
the script independently in several terminals.

``` python
import os

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.data.distributed import DistributedSampler


class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.network(x)


def main():
    # torchrun supplies LOCAL_RANK, RANK and WORLD_SIZE.
    local_rank = int(os.environ["LOCAL_RANK"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    # NCCL is the usual backend for NVIDIA GPU training.
    dist.init_process_group(backend="nccl")

    try:
        model = Model().to(device)
        model = DDP(model, device_ids=[local_rank])

        # Example dataset. In real training, replace this with your dataset.
        features = torch.randn(4096, 1024)
        targets = torch.randint(0, 10, (4096,))
        dataset = TensorDataset(features, targets)

        sampler = DistributedSampler(dataset, shuffle=True)
        loader = DataLoader(
            dataset,
            batch_size=32,
            sampler=sampler,
            num_workers=0,
            pin_memory=True,
        )

        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(3):
            sampler.set_epoch(epoch)
            model.train()

            for inputs, labels in loader:
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # DDP synchronizes gradients as backward executes.
                loss.backward()

                # Each rank applies the same synchronized gradient update.
                optimizer.step()

            if dist.get_rank() == 0:
                print(f"Finished epoch {epoch}")

    finally:
        dist.destroy_process_group()


if __name__ == "__main__":
    main()
```

Run on a machine with four available CUDA GPUs:

``` bash
torchrun --standalone --nproc_per_node=4 ddp_train.py
```

For two GPUs:

``` bash
torchrun --standalone --nproc_per_node=2 ddp_train.py
```

The `--nproc_per_node` value is the number of processes to launch on the
machine. In the common one-process-per-GPU setup, it is also the number
of GPUs used.

## 4. Important DDP APIs

  -----------------------------------------------------------------------
  API                                 Purpose
  ----------------------------------- -----------------------------------
  `dist.init_process_group()`         Initializes distributed
                                      communication among processes.

  `DistributedDataParallel()`         Wraps a model to perform
                                      distributed gradient
                                      synchronization.

  `DistributedSampler()`              Assigns a portion of the dataset to
                                      each process.

  `sampler.set_epoch(epoch)`          Changes the sampler's shuffle order
                                      consistently between epochs.

  `loss.backward()`                   Computes gradients and triggers
                                      DDP's gradient synchronization.

  `dist.all_reduce()`                 Explicitly reduces a tensor across
                                      processes when needed. DDP normally
                                      handles model-gradient reduction
                                      itself.

  `dist.barrier()`                    Makes processes wait at a common
                                      synchronization point.

  `dist.destroy_process_group()`      Cleans up the distributed process
                                      group.
  -----------------------------------------------------------------------

## 5. DDP synchronization: what happens during one iteration?

Assume four processes, each with a model replica and a local batch of 32
samples.

-   Each process performs a forward pass independently.
-   Each process calculates its local loss and gradients.
-   During backward propagation, DDP groups gradients into buckets and
    communicates them using collective operations such as All-Reduce.
-   The gradients are reduced across ranks. By default, DDP averages the
    gradients across the participating processes.
-   Each process calls `optimizer.step()` using the synchronized
    gradients.
-   With matching optimizer state and update logic, each replica reaches
    the same parameter values.

**Important:** DDP synchronizes gradients, not the full model parameters
after every optimizer step. It relies on consistent initialization and
identical updates on all ranks.

## 6. DDP vs. other synchronization techniques

  -----------------------------------------------------------------------
  Technique                           What it synchronizes or protects
  ----------------------------------- -----------------------------------
  `threading.Lock()`                  Protects shared resources between
                                      threads in one process.

  `torch.multiprocessing.Lock()`      Protects shared resources accessed
                                      by multiple processes.

  `threading.Barrier()`               Waits until all participating
                                      threads reach a point.

  `dist.barrier()`                    Waits until all participating
                                      distributed processes reach a
                                      point.

  `torch.cuda.synchronize()`          Makes the calling CPU thread wait
                                      for previously submitted work on
                                      the selected CUDA device.

  DDP gradient All-Reduce             Aggregates gradients across
                                      distributed training processes.
  -----------------------------------------------------------------------

These mechanisms are not interchangeable. A lock protects a critical
section; a barrier coordinates arrival at a point; CUDA synchronization
waits for GPU work; DDP coordinates distributed gradient updates.

## 7. Common DDP issues

### A. One process hangs during backward

A rank may fail or skip a collective operation while other ranks are
waiting for it. Ensure all ranks execute compatible forward and backward
paths and participate in the same collectives.

### B. Incorrect data distribution

Use `DistributedSampler` so each process receives its intended portion
of the dataset. Call `sampler.set_epoch(epoch)` before each epoch when
shuffling.

### C. Different model updates across ranks

Ensure each rank uses the same optimizer configuration and performs
compatible update steps. Avoid rank-specific changes to model parameters
unless deliberately coordinated.

### D. Slow synchronization

A slow GPU, unbalanced workload, or communication bottleneck can make
faster ranks wait. Profile both computation and communication.

### E. Confusing `dist.barrier()` with gradient synchronization

`dist.barrier()` only coordinates process arrival. It does not average
gradients. DDP's gradient communication is handled automatically during
backward propagation.

## 8. Interview questions

**Q1. Does DDP synchronize model parameters after every forward pass?**

No. DDP synchronizes gradients during backward propagation. Each rank
then updates its local model using those synchronized gradients.

**Q2. What happens if one GPU is slower than another?**

Other ranks may wait for the slower rank at collective communication
points. This can reduce overall training throughput.

**Q3. Does DDP synchronize gradients or model weights?**

DDP synchronizes gradients through collective communication, typically
All-Reduce. Model replicas remain aligned because they start
consistently and apply the same updates.

**Q4. Is `dist.barrier()` required after every training iteration?**

No. DDP already coordinates gradient synchronization. Adding barriers
unnecessarily can reduce performance.

**Q5. Do I need to call `dist.all_reduce()` manually for model
gradients?**

Normally, no. DDP performs gradient synchronization automatically.
Manual collectives are useful for additional distributed values or
custom training logic.

## 9. Interview takeaway

**Distributed Data Parallel (DDP)** trains model replicas across
multiple GPUs. Each process computes gradients using its local
mini-batch. DDP uses All-Reduce to aggregate and average gradients
across processes during backward propagation. Each process then applies
the same optimizer update, keeping model replicas synchronized while
allowing computation to run in parallel.


## Semaphore Synchronizations in PyTorch

TODO

# 10 Real-World CUDA Deadlock & Synchronization Debugging Scenarios in PyTorch
> **Important:** A CUDA hang is not always a true deadlock. It may also be caused by an infinite kernel, an unfulfilled synchronization dependency, a blocked CPU thread, or a long-running GPU operation.

## 1. Asynchronous CUDA execution

### Problem

The CPU continues executing before the GPU finishes its operation.

```python
import torch
import time

device = "cuda"

x = torch.randn(5000, 5000, device=device)
y = torch.randn(5000, 5000, device=device)

start = time.time()
z = torch.matmul(x, y)

print("Execution time:", time.time() - start)
```

### Why does this happen?

- PyTorch launches CUDA operations asynchronously.
- CPU execution continues without waiting for the GPU.
- The measured time may represent kernel launch overhead rather than actual GPU execution.

### Debugging solution

```python
torch.cuda.synchronize()

start = time.time()
z = torch.matmul(x, y)
torch.cuda.synchronize()

print("Execution time:", time.time() - start)
```

**Interview takeaway:** `torch.cuda.synchronize()` waits for previously queued GPU work on the current device to complete.

---

## 2. Race condition between CUDA streams

**Category:** Race condition | Intermediate

### Problem

Stream 2 reads a tensor before Stream 1 finishes updating it.

```python
import torch

stream1 = torch.cuda.Stream()
stream2 = torch.cuda.Stream()

x = torch.zeros(1000, device="cuda")

with torch.cuda.stream(stream1):
    x.fill_(10)

with torch.cuda.stream(stream2):
    y = x * 2

torch.cuda.synchronize()
print(y[:5])
```

There is no dependency between the two streams. The result may be incorrect or nondeterministic.

### Solution: Use CUDA events

```python
event = torch.cuda.Event()

with torch.cuda.stream(stream1):
    x.fill_(10)
    event.record()

with torch.cuda.stream(stream2):
    stream2.wait_event(event)
    y = x * 2

torch.cuda.synchronize()
print(y[:5])
```

Expected result:

```text
tensor([20., 20., 20., 20., 20.], device='cuda:0')
```

**Interview takeaway:** CUDA events establish dependencies between streams without requiring the CPU to block.

---

## 3. Tensor memory lifetime issue with custom streams

**Category:** Memory safety | Intermediate

### Problem

A tensor is created in one stream and consumed in another. Its memory may be reused by PyTorch's caching allocator before the second stream finishes using it.

### Example

```python
import torch

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

with torch.cuda.stream(s1):
    x = torch.randn(1000, device="cuda")
    y = x * 2

with torch.cuda.stream(s2):
    s2.wait_stream(s1)
    y.record_stream(s2)
    z = y * 3

torch.cuda.synchronize()
print(z[:5])
```
#### Why record_stream()?
`y.record_stream(s2)` is required because the tensor `y` is created and produced in **stream s1**, but later it is consumed in **stream s2** during the operation `z = y * 3`. PyTorch's CUDA memory allocator tracks the lifetime of GPU memory associated with tensors and normally assumes that the tensor is only being used by the stream that created it. If `y` is used in **s2** without calling `y.record_stream(s2)`, the allocator may incorrectly think that the memory occupied by `y` is no longer needed after **s1** finishes, and it could reuse that memory while operations in **s2** are still accessing `y`. By calling `y.record_stream(s2)`, we inform PyTorch that the tensor `y` will continue to be used in **stream s2**, so the allocator keeps the memory alive until all work involving `y` in **s2** has completed. Thus, `record_stream()` ensures correct GPU memory lifetime management when a tensor is shared across multiple CUDA streams and prevents premature memory reuse that could lead to incorrect results or memory corruption.


### What do these functions do?

- `wait_stream()` establishes execution ordering.
- `record_stream()` informs the `caching allocator `that the tensor is still being used by another stream.
- `synchronize()` waits for completion.

**Interview takeaway:** Stream synchronization and tensor memory lifetime management are two separate problems.

---

## 4. Excessive CPU-GPU synchronization

**Category:** Performance issue | Beginner

### Problem

Calling `.item()` repeatedly during training forces the CPU to obtain a GPU result.

```python
import torch

x = torch.randn(1000, 1000, device="cuda")

for i in range(100):
    loss = (x * x).mean()
    value = loss.item()
    print(value)
```

### Why is this inefficient?

1. GPU calculates the loss.
2. CPU requests the scalar value.
3. CPU waits for the required GPU work to finish.
4. The next iteration proceeds.

### Optimization

```python
loss_values = []

for i in range(100):
    loss = (x * x).mean()
    loss_values.append(loss.detach())

torch.cuda.synchronize()
values = torch.stack(loss_values).cpu()

print(values)
```
`torch.stack(loss_values).cpu()` :

This line does **two things**. First, `torch.stack(loss_values)` converts the Python list `loss_values`, which contains 100 individual scalar tensors collected from `loss.detach()`, into a single tensor. For example, 
- if:
    ```python
    loss_values = [tensor(2.1), tensor(1.8), tensor(1.5)]
    ```
- then
    ```python
    torch.stack(loss_values)
    ```
- becomes
    ```python
    tensor([2.1, 1.8, 1.5])
    ```
**Second**, `.cpu()` moves this tensor from GPU memory to CPU memory. Since `loss` was computed from `x` (which may be on CUDA), the stored losses are also on the GPU. Moving them to the CPU makes them easier to print, save, convert to NumPy, or analyze later. So `values` becomes a single CPU tensor containing all 100 loss values:


**Interview takeaway:** Avoid unnecessary `.item()`, `.cpu()`, and explicit synchronization inside performance-critical training loops.

---

## 5. GPU hangs during training

**Category:** GPU hang | Intermediate

### Problem

Training appears stuck during forward or backward execution.

```python
import torch
import torch.nn as nn

device = "cuda"

model = nn.Linear(1000, 1000).to(device)
optimizer = torch.optim.Adam(model.parameters())

for epoch in range(100):
    x = torch.randn(1000, 1000, device=device)

    optimizer.zero_grad()
    output = model(x)
    loss = output.mean()

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch)
```

### Debugging approach

Add synchronization points to isolate the operation.

```python
for epoch in range(100):
    x = torch.randn(1000, 1000, device=device)

    optimizer.zero_grad()

    output = model(x)
    torch.cuda.synchronize()
    print("Forward completed")

    loss = output.mean()

    loss.backward()
    torch.cuda.synchronize()
    print("Backward completed")

    optimizer.step()
    torch.cuda.synchronize()
    print("Optimizer completed")

    print("Epoch:", epoch)
```

If the program stops before printing `Backward completed`, investigate the backward operation and its CUDA kernels.

Also try:

```bash
CUDA_LAUNCH_BLOCKING=1 python train.py
```

**Important:** Synchronization points help locate the failure but change execution timing. Remove them after debugging.

---

## 6. Incorrect thread barrier using `__syncthreads()`

**Category:** Kernel synchronization | Advanced

This is a CUDA C++ kernel-level problem, not something directly exposed by ordinary PyTorch tensor operations.

### Problem

Some threads reach a synchronization barrier while others skip it.

### Incorrect CUDA kernel

```cpp
__global__ void faulty_kernel(float* data)
{
    int tid = threadIdx.x;

    if (tid < 16)
    {
        data[tid] = tid;
        __syncthreads();
    }
}
```

Only some threads reach `__syncthreads()`. This violates the block-wide synchronization requirement and can cause undefined behavior or a kernel hang.

### Correct approach

```cpp
__global__ void correct_kernel(float* data)
{
    int tid = threadIdx.x;

    if (tid < 16)
    {
        data[tid] = tid;
    }

    __syncthreads();

    // All threads reach this barrier.
}
```

### What does `__syncthreads()` do?

- Synchronizes threads within one thread block.
- Ensures participating threads reach the barrier.
- Makes prior shared-memory accesses visible to threads in that block.
- Does not synchronize different thread blocks.

**Interview takeaway:** Never place a block-wide barrier inside divergent control flow unless all threads in the block are guaranteed to reach it.

---


## 7. CUDA multiprocessing deadlock

**Category:** Deadlock risk | Advanced

### Problem

A PyTorch application hangs when CUDA is initialized before creating multiprocessing workers using the `fork` start method.

This is commonly associated with CUDA's inability to safely reinitialize after a process fork.

### Potentially problematic approach

```python
import torch
import multiprocessing as mp

torch.cuda.init()

def worker():
    x = torch.randn(100, device="cuda")
    print(x.sum())

if __name__ == "__main__":
    p = mp.Process(target=worker)
    p.start()
    p.join()
```

### Solution: Use `spawn`

```python
import torch
import torch.multiprocessing as mp

def worker(rank):
    torch.cuda.set_device(rank)

    x = torch.randn(100, device=f"cuda:{rank}")
    print("Worker:", rank, x.sum().item())

if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    p = mp.Process(target=worker, args=(0,))
    p.start()
    p.join()
```

**Interview takeaway:** Use `spawn` or another CUDA-safe multiprocessing strategy instead of initializing CUDA in a parent and then forking CUDA-using children.

---





## 8. DistributedDataParallel (DDP) deadlock

**Category:** Distributed training | Advanced

### Problem

One GPU waits indefinitely for another GPU during gradient synchronization.

DDP uses collective communication operations, commonly through NCCL.

For example:

- GPU 0 executes an all-reduce operation.
- GPU 1 never reaches the corresponding collective.
- GPU 0 continues waiting.
- Training hangs.

### Example of problematic logic

```python
if rank == 0:
    loss.backward()

optimizer.step()
```

If the model uses DDP, rank 1 also needs to participate in the required backward gradient synchronization.

### Correct approach

```python
output = ddp_model(inputs)
loss = criterion(output, targets)

optimizer.zero_grad()
loss.backward()
optimizer.step()
```

Both ranks should execute compatible collective operations in the same order.

### Debugging configuration

```bash
export TORCH_DISTRIBUTED_DEBUG=DETAIL
export NCCL_DEBUG=INFO
export NCCL_ASYNC_ERROR_HANDLING=1

torchrun --nproc_per_node=2 train.py
```

**Interview takeaway:** DDP deadlocks commonly result from mismatched collective operations, inconsistent control flow, or a rank failing before reaching a collective.

---

## 9. Circular dependency between CUDA streams

**Category:** Deadlock scenario | Advanced

### Problem

Two CUDA streams wait for events that depend on each other's execution.

Conceptual dependency:

```text
Stream 1 waits for Event B
        |
        v
Stream 2 waits for Event A
        |
        +---- Circular dependency
```

### Problematic code

```python
import torch

s1 = torch.cuda.Stream()
s2 = torch.cuda.Stream()

event_a = torch.cuda.Event()
event_b = torch.cuda.Event()

with torch.cuda.stream(s1):
    s1.wait_event(event_b)
    event_a.record()

with torch.cuda.stream(s2):
    s2.wait_event(event_a)
    event_b.record()

torch.cuda.synchronize()
```

This creates a circular dependency between the streams and can cause a GPU-side wait cycle.

### Solution: Establish a one-directional dependency

```python
with torch.cuda.stream(s1):
    x = torch.randn(1000, device="cuda")
    event_a.record()

with torch.cuda.stream(s2):
    s2.wait_event(event_a)
    y = x * 2

torch.cuda.synchronize()
```

**Interview takeaway:** Avoid cyclic dependencies between CUDA streams. Design execution as a directed dependency graph.

---

## 10. CUDA kernel execution failure mistaken for a deadlock

**Category:** GPU debugging | Practical

### Problem

Training appears to hang, but the actual issue is an illegal memory access or another asynchronous CUDA kernel error.

### Example: Invalid indexing

```python
import torch

x = torch.randn(100, device="cuda")

indices = torch.tensor(
    [10, 20, 200],
    device="cuda"
)

y = x[indices]

torch.cuda.synchronize()
```

Index `200` is outside the valid range of the tensor. This produces an indexing error rather than a genuine deadlock.

### Debugging solution

Run:

```bash
CUDA_LAUNCH_BLOCKING=1 python train.py
```

For custom CUDA kernels, use NVIDIA Compute Sanitizer:

```bash
compute-sanitizer --tool memcheck python train.py
```

Other useful commands:

```bash
nvidia-smi
nvidia-smi dmon
watch -n 1 nvidia-smi
```

### What should you investigate?

- GPU utilization stuck at 0%.
- GPU utilization stuck at 100%.
- GPU memory continuously increasing.
- Kernel execution errors.
- CUDA processes waiting indefinitely.
- GPU execution time increasing unexpectedly.

**Interview takeaway:** A GPU hang is a symptom, not a diagnosis. First distinguish synchronization problems, kernel errors, memory problems, and genuinely long-running operations.

---

## 11. Summary: All 10 debugging scenarios

| # | Scenario | Main debugging technique |
|---:|---|---|
| 1 | Asynchronous CUDA execution | `torch.cuda.synchronize()` |
| 2 | Race condition between streams | CUDA events |
| 3 | Tensor memory lifetime issue | `record_stream()` |
| 4 | Excessive CPU-GPU synchronization | Avoid repeated `.item()` |
| 5 | GPU hangs during training | Synchronization checkpoints |
| 6 | Incorrect thread barriers | `__syncthreads()` |
| 7 | Multiprocessing deadlock | `spawn` |
| 8 | DDP deadlock | NCCL debugging |
| 9 | Circular stream dependency | Correct event ordering |
| 10 | Kernel execution failure | Compute Sanitizer |

---

## 12. Practical CUDA debugging commands cheat sheet

| Command / API | Usage |
|---|---|
| `torch.cuda.synchronize()` | Wait for GPU completion |
| `torch.cuda.current_stream()` | Get current CUDA stream |
| `torch.cuda.Stream()` | Create CUDA stream |
| `torch.cuda.Event()` | Create CUDA event |
| `stream.wait_event()` | Wait for event |
| `stream.wait_stream()` | Wait for another stream |
| `tensor.record_stream()` | Track tensor usage on a stream |
| `CUDA_LAUNCH_BLOCKING=1` | Synchronous CUDA debugging |
| `compute-sanitizer` | Detect CUDA memory and synchronization errors |
| `nvidia-smi` | Inspect GPU status |

---

## 13. CUDA debugging interview questions

1. Why does PyTorch execute CUDA operations asynchronously?
2. Which API establishes a dependency between two CUDA streams?
3. What is the purpose of `tensor.record_stream()`?
4. What can cause a DDP training hang?
5. What does `CUDA_LAUNCH_BLOCKING=1` do?
6. What is the difference between `torch.cuda.synchronize()` and `stream.synchronize()`?
7. How can incorrect use of `__syncthreads()` cause a kernel hang?
8. How do CUDA events help prevent race conditions?
9. Why should CUDA multiprocessing generally use `spawn` instead of `fork`?
10. How would you distinguish a deadlock from an asynchronous CUDA error?

---

## 14. Recommended next topics

For advanced CUDA debugging interviews, study these topics next:

1. Warp-level synchronization and divergence.
2. Shared memory bank conflicts.
3. Global memory race conditions.
4. Atomic operations (`atomicAdd`, `atomicCAS`).
5. CUDA memory fences.
6. Stream priorities and concurrent kernel execution.
7. CUDA Graph synchronization.
8. NCCL communication deadlocks.

These topics take you from PyTorch-level debugging to understanding what actually happens inside CUDA kernels on NVIDIA GPUs.
